# Practical 8: Stock Market Prediction using Recurrent Neural Networks

**Problem Statement:** Predict stock price trends using historical NASDAQ stock data and RNN/LSTM models.

**Activities:**
1. Prepare sequential data
2. Implement LSTM network
3. Visualize predicted vs actual values

**Dataset:** Historical daily closing prices for AAPL (NASDAQ), downloaded via yfinance.

## 1. Import Libraries and Download Stock Data

In [ ]:
!pip install -q yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')
df = df[['Close']]
df.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df['Close'])
plt.title('AAPL Closing Price History')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.show()

## 2. Prepare Sequential Data

Closing prices are scaled to [0, 1] and reorganized into overlapping sequences: each input is a window of 60 past days, and the target is the closing price on the following day.

In [ ]:
scaler = MinMaxScaler()
scaled_close = scaler.fit_transform(df[['Close']])

WINDOW_SIZE = 60

def create_sequences(data, window_size):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_close, WINDOW_SIZE)
X = X.reshape(X.shape[0], X.shape[1], 1)

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 3. Implement LSTM Network

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(WINDOW_SIZE, 1)),
    keras.layers.LSTM(64, return_sequences=True),
    keras.layers.LSTM(64),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)
])

model.summary()

## 4. Model Training

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=32,
    verbose=1
)

## 5. Predict and Evaluate

In [ ]:
y_pred_scaled = model.predict(X_test, verbose=0)

y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
print(f"Test RMSE: {rmse:.4f}")

## 6. Visualize Predicted vs Actual Values

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_test_actual, label='Actual Price')
plt.plot(y_pred, label='Predicted Price')
plt.title('AAPL Stock Price: Predicted vs Actual')
plt.xlabel('Time Step')
plt.ylabel('Closing Price (USD)')
plt.legend()
plt.show()

## Conclusion

In this practical, we:
- Downloaded historical NASDAQ (AAPL) stock data
- Prepared sequential data using a sliding window approach
- Implemented and trained an LSTM network for price prediction
- Evaluated performance using RMSE and visualized predicted vs actual prices